# SEED 被试内 EEGNet 验证基线

这份代码用的是同一被试的一次 Session。我按完整 Trial 划分数据，6 个训练、3 个验证、6 个测试。每个窗口是 1 秒，输入形状为 `(B, 1, 62, 200)`。

训练默认运行 50 轮，每轮做验证并保存验证交叉熵最低的模型。训练记录和权重会保存到 Google Drive 的 `EEG_Research/results/EEGNet/`。我先看训练曲线，确定设置后再运行最终测试。

每个 Trial 都用自己的整段信号统计量做标准化，因此这里按已完整采集的离线 EEG 处理。之前已经查看过的测试 Trial 只用于探索性对照。

## 1. 挂载 Google Drive 和设置参数

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import random
import re
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/EEG_Research/datasets/SEED/Preprocessed_EEG")
OUTPUT_ROOT = Path("/content/drive/MyDrive/EEG_Research/results/EEGNet")

SUBJECT_ID = 1
SESSION_INDEX = 0       # 先按文件名排序，0 取第一份记录
RANDOM_SEED = 42
EPOCHS = 50             # 确定需要延长训练时，可在这里改为 500
WINDOW_POINTS = 200     # SEED 预处理信号为 200 Hz，200 点为 1 秒
NUM_ELECTRODES = 62
NUM_CLASSES = 3
CLASS_NAMES = ("negative", "neutral", "positive")

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前设备", torch.cuda.get_device_name(0) if device.type == "cuda" else device)

## 2. 定义 EEGNet

这个模型使用 EEGNet 的时空卷积结构，并设置最大范数约束。我先用一个虚拟输入计算分类层需要的特征数，计算时将特征层切到评估模式。训练时，每次 Adam 更新后会约束权重。

In [ ]:
class EEGNetOfficialPyTorch(nn.Module):
    def __init__(
        self,
        num_classes=3,
        num_electrodes=62,
        chunk_size=200,
        F1=8,
        D=2,
        F2=16,
        kernel_length=100,
        dropout_rate=0.5,
    ):
        super().__init__()
        spatial_channels = F1 * D

        self.block1 = nn.Sequential(
            nn.Conv2d(
                1, F1, kernel_size=(1, kernel_length),
                padding=(0, kernel_length // 2), bias=False,
            ),
            nn.BatchNorm2d(F1, momentum=0.01, eps=1e-3),
            nn.Conv2d(
                F1, spatial_channels, kernel_size=(num_electrodes, 1),
                groups=F1, bias=False,
            ),
            nn.BatchNorm2d(spatial_channels, momentum=0.01, eps=1e-3),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4), stride=(1, 4)),
            nn.Dropout(dropout_rate),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(
                spatial_channels, spatial_channels, kernel_size=(1, 16),
                padding=(0, 8), groups=spatial_channels, bias=False,
            ),
            nn.Conv2d(spatial_channels, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2, momentum=0.01, eps=1e-3),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8), stride=(1, 8)),
            nn.Dropout(dropout_rate),
        )

        # 试算分类头的输入大小，同时让 BatchNorm 处于评估模式。
        self.block1.eval()
        self.block2.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_electrodes, chunk_size)
            self.feature_dim = self.extract_features(dummy).numel()
        self.block1.train()
        self.block2.train()

        self.classifier = nn.Linear(self.feature_dim, num_classes)
        self.project_weights()

    @torch.no_grad()
    def project_weights(self):
        # dim=0 对空间卷积逐输出核、对分类头逐类别权重行约束。
        for layer, limit in ((self.block1[2], 1.0), (self.classifier, 0.25)):
            projected = torch.renorm(layer.weight, p=2, dim=0, maxnorm=limit)
            layer.weight.copy_(projected)

    def extract_features(self, x):
        return self.block2(self.block1(x))

    def forward(self, x):
        features = self.extract_features(x)
        return self.classifier(features.flatten(start_dim=1))

## 3. 读取 Session 并按完整 Trial 划分

在 Trial 1–9 中，我从每类各选最后出现的 1 个做验证，剩下的 6 个用于训练。Trial 10–15 到最终测试单元格才会切窗并构建批次。

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"找不到 SEED 文件夹，请检查 DATA_DIR：{DATA_DIR}")

label_file = DATA_DIR / "label.mat"
if not label_file.is_file():
    raise FileNotFoundError(f"找不到标签文件：{label_file}")

raw_labels = sio.loadmat(label_file)["label"].reshape(-1)
if len(raw_labels) != 15 or set(raw_labels.tolist()) != {-1, 0, 1}:
    raise ValueError("预计 label.mat 含 15 个 Trial 标签，类别为 -1、0、1。")
labels = (raw_labels + 1).astype(np.int64)

session_files = sorted(DATA_DIR.glob(f"{SUBJECT_ID}_*.mat"))
if not 0 <= SESSION_INDEX < len(session_files):
    raise FileNotFoundError(
        f"被试 {SUBJECT_ID} 找到 {len(session_files)} 个 Session，"
        f"请检查 SUBJECT_ID 和 SESSION_INDEX。"
    )
session_file = session_files[SESSION_INDEX]
mat_data = sio.loadmat(session_file)

trial_keys = {}
for key in mat_data:
    suffix = re.search(r"_eeg(\d+)$", key)
    if suffix:
        trial_id = int(suffix.group(1))
        if trial_id in trial_keys:
            raise ValueError(f"发现重复的 Trial 编号 {trial_id}。")
        trial_keys[trial_id] = key

if sorted(trial_keys) != list(range(1, 16)):
    raise ValueError(f"应找到 Trial 1–15，实际找到 {sorted(trial_keys)}。")

# 编号从 1 开始以便阅读，取 NumPy 标签时减 1。
training_pool = list(range(1, 10))
test_ids = list(range(10, 16))
val_ids = []
for class_id in range(NUM_CLASSES):
    candidates = [trial_id for trial_id in training_pool if labels[trial_id - 1] == class_id]
    if len(candidates) < 2:
        raise ValueError(f"前 9 个 Trial 中，类别 {class_id} 至少需要 2 个 Trial。")
    val_ids.append(candidates[-1])

val_ids.sort()
train_ids = [trial_id for trial_id in training_pool if trial_id not in val_ids]

print("读取 Session", session_file.name)
print("训练 Trial", train_ids, "标签", [int(labels[i - 1]) for i in train_ids])
print("验证 Trial", val_ids, "标签", [int(labels[i - 1]) for i in val_ids])
print("留出测试 Trial", test_ids)


def build_trials(trial_ids):
    windows, targets = [], []
    for trial_id in trial_ids:
        trial = mat_data[trial_keys[trial_id]]
        if trial.ndim != 2 or trial.shape[0] != NUM_ELECTRODES:
            raise ValueError(f"Trial {trial_id} 的形状是 {trial.shape}，应为 (62, T)。")
        if not np.isfinite(trial).all():
            raise ValueError(f"Trial {trial_id} 包含 NaN 或 Inf。")

        num_windows = trial.shape[1] // WINDOW_POINTS
        if num_windows == 0:
            raise ValueError(f"Trial {trial_id} 没有足够数据切出一个完整窗口。")

        mean = trial.mean(axis=1, keepdims=True)
        std = trial.std(axis=1, keepdims=True)
        normalized = (trial - mean) / (std + 1e-8)
        for window_index in range(num_windows):
            start = window_index * WINDOW_POINTS
            chunk = normalized[:, start : start + WINDOW_POINTS]
            windows.append(chunk[np.newaxis, :, :])
            targets.append(labels[trial_id - 1])

    return np.array(windows, dtype=np.float32), np.array(targets, dtype=np.int64)


def make_loader(trial_ids, batch_size, shuffle=False):
    x, y = build_trials(trial_ids)
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    generator = torch.Generator().manual_seed(RANDOM_SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)

## 4. 训练并用验证集选权重

每轮计算验证交叉熵，验证损失降低时保存权重。测试集不进入这段训练循环。

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

train_loader = make_loader(train_ids, TRAIN_BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_ids, EVAL_BATCH_SIZE)
print("训练窗口数", len(train_loader.dataset), "| 验证窗口数", len(val_loader.dataset))

model = EEGNetOfficialPyTorch(
    num_classes=NUM_CLASSES,
    num_electrodes=NUM_ELECTRODES,
    chunk_size=WINDOW_POINTS,
).to(device)
print("模型参数量", f"{sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def train_one_epoch(model, loader, optimizer):
    model.train()
    loss_sum = 0.0
    count = 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        model.project_weights()

        loss_sum += loss.item() * by.size(0)
        count += by.size(0)

    return loss_sum / count


@torch.no_grad()
def evaluate(model, loader, collect_confusion=False):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        logits = model(bx)
        preds = logits.argmax(dim=1)
        loss_sum += criterion(logits, by).item() * by.size(0)
        correct += (preds == by).sum().item()
        count += by.size(0)
        if collect_confusion:
            np.add.at(confusion, (by.cpu().numpy(), preds.cpu().numpy()), 1)

    return loss_sum / count, 100.0 * correct / count, confusion


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = Path(tempfile.mkdtemp(prefix=f"{session_file.stem}_{run_stamp}_", dir=OUTPUT_ROOT))
checkpoint_path = run_dir / "best_eegnet.pt"
history_path = run_dir / "history.csv"

best_state = None
best_val_loss = float("inf")
best_epoch = 0
history = []

with history_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["epoch", "train_loss", "val_loss", "val_acc"])
    writer.writeheader()

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc, _ = evaluate(model, val_loader)
        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise RuntimeError(f"第 {epoch} 轮损失异常，请检查数据和训练设置。")

        entry = dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc)
        history.append(entry)
        writer.writerow(entry)
        stream.flush()

        improved = val_loss < best_val_loss
        if improved:
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            torch.save(
                {
                    "model_state_dict": best_state,
                    "epoch": best_epoch,
                    "val_loss": best_val_loss,
                    "val_acc": val_acc,
                    "session_file": session_file.name,
                    "train_trials": train_ids,
                    "validation_trials": val_ids,
                    "test_trials": test_ids,
                    "window_points": WINDOW_POINTS,
                    "seed": RANDOM_SEED,
                },
                checkpoint_path,
            )

        marker = " | 保存最佳权重" if improved else ""
        print(
            f"Epoch {epoch:03d}/{EPOCHS} | 训练 Loss {train_loss:.4f} | "
            f"验证 Loss {val_loss:.4f} | 验证准确率 {val_acc:.2f}%{marker}"
        )

print(f"验证 Loss 最低的轮次为第 {best_epoch} 轮，Loss {best_val_loss:.4f}。")
print("最佳权重文件", checkpoint_path)
print("训练日志文件", history_path)

## 5. 看训练曲线

先比较训练损失和验证损失，再决定下一次是否延长训练。

In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, [row["train_loss"] for row in history], label="Train")
axes[0].plot(epochs, [row["val_loss"] for row in history], label="Validation")
axes[0].axvline(best_epoch, color="gray", linestyle="--", label=f"Selected epoch {best_epoch}")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy")
axes[0].legend()

axes[1].plot(epochs, [row["val_acc"] for row in history])
axes[1].set(xlabel="Epoch", ylabel="Validation window accuracy (%)")

fig.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=150)
plt.show()

## 6. 最终测试

训练设置确定后运行这格。代码会加载验证损失最低时的权重，再对后 6 个 Trial 的窗口做评估。混淆矩阵的行是真实类别，列是预测类别。

In [ ]:
if best_state is None:
    raise RuntimeError("请先运行训练与验证单元格。")

model.load_state_dict(best_state)
test_loader = make_loader(test_ids, EVAL_BATCH_SIZE)
test_loss, test_acc, test_confusion = evaluate(model, test_loader, collect_confusion=True)

print("使用的模型轮次", best_epoch)
print("测试窗口数", len(test_loader.dataset))
print(f"测试 Loss {test_loss:.4f} | 窗口准确率 {test_acc:.2f}%")
print("类别顺序", CLASS_NAMES)
print("混淆矩阵")
print(test_confusion)

with (run_dir / "test_results.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["selected_epoch", "test_loss", "test_window_acc_percent", "test_window_count"])
    writer.writerow([best_epoch, test_loss, test_acc, len(test_loader.dataset)])

with (run_dir / "test_confusion_matrix.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["true / predicted", *CLASS_NAMES])
    for label, counts in zip(CLASS_NAMES, test_confusion.tolist()):
        writer.writerow([label, *counts])

print("全部运行记录", run_dir)